<a href="https://colab.research.google.com/github/Saarss-2211/projects/blob/main/SAKT_%2B_Dueling_DQN_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
# ===== 1. Imports =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# Extract Dataset

In [ ]:
# ===== Extract Dataset =====
import zipfile
import os

# Unzip the file
with zipfile.ZipFile('/content/may_to_june.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

# List the contents of the extracted folder
print("files extracted:",os.listdir('data'))

files extracted: ['alogs.csv', 'pdets.csv', 'tlogs.csv', 'cdets.csv', 'ddets.csv', 'tdets.csv', 'adets.csv', 'sdets.csv', 'slogs.csv', 'plogs.csv']


In [ ]:
import pandas as pd

plogs = pd.read_csv('data/plogs.csv')

display(plogs.head())

,log_id,student_id,assignment_id,problem_id,start_time,time_on_task,answer_before_tutoring,fraction_of_hints_used,attempt_count,answer_given,problem_completed,correct
0,16062814,740890,1397084,1220201,2021-05-01 13:48:08.687000+00:00,29.111,True,0.0,1,False,False,False
1,16062884,802471,1399980,1762838,2021-05-01 14:50:45.901000+00:00,35.779,True,NaN,1,False,True,True
2,16062884,802471,1399980,1762839,2021-05-01 14:51:22.675000+00:00,151.948,True,NaN,1,False,True,True
3,16062884,802471,1399980,1762840,2021-05-01 14:53:55.110000+00:00,4.658,True,NaN,1,False,True,True
4,16062884,802471,1399980,1762841,2021-05-01 14:54:00.211000+00:00,142.534,True,NaN,1,False,True,NaN


In [ ]:
import pandas as pd

pdets = pd.read_csv('data/pdets.csv')

display(pdets.head())

,problem_id,content_source,skills,problem_type,tutoring_types,student_answer_count,mean_correct,mean_time_on_task
0,33,['Certified Content'],['8.NS.A.2-1'],Multiple Choice,['Hint'],8,0.625000,NaN
1,35,['Certified Content'],['8.NS.A.2-1'],Exact Match (case sensitive),['Hint'],5,0.200000,NaN
2,37,['Certified Content'],['8.NS.A.2-1'],Exact Match (case sensitive),['Hint'],3,0.666667,NaN
3,39,['Certified Content'],['8.NS.A.2-1'],Multiple Choice,['Hint'],3,1.000000,NaN
4,117,['Certified Content'],['8.NS.A.2-1'],Multiple Choice,['Scaffold'],15,1.000000,102.052267


# Load Required Columns Only

In [ ]:
# ===== Load Required Columns Only =====

print("PLOGS shape:", plogs.shape)
print("PDETS shape:", pdets.shape)

PLOGS shape: (3249071, 12)
PDETS shape: (80898, 8)


In [ ]:
data = plogs.merge(pdets, on="problem_id", how="inner")

print("After merge:", data.shape)

After merge: (3249071, 19)


In [ ]:
data = data.dropna(subset=["skills"])

print("After removing NaN skills:", data.shape)

After removing NaN skills: (1474670, 19)


In [ ]:
data["skills"].head()

,skills
0,['2.NBT.B.7']
11,"['6.SP.B.4-6', '6.SP.B.5c-1', '6.SP.B.5c-2', '..."
12,"['6.SP.B.4-6', '6.SP.B.5c-1', '6.SP.B.5c-2', '..."
13,"['6.SP.B.4-6', '6.SP.B.5c-1', '6.SP.B.5c-2', '..."
14,"['6.SP.B.4-6', '6.SP.B.5c-1', '6.SP.B.5c-2', '..."


In [ ]:
import ast

def extract_first_skill(skill_str):
    try:
        skill_list = ast.literal_eval(skill_str)
        if isinstance(skill_list, list) and len(skill_list) > 0:
            return skill_list[0]
    except:
        return None
    return None

data["skill"] = data["skills"].apply(extract_first_skill)

data = data.dropna(subset=["skill"])

print("After skill extraction:", data.shape)

After skill extraction: (1474670, 20)


In [ ]:
data = data.dropna(subset=["correct"])   # remove NaN correctness
data["correct"] = data["correct"].astype(int)

# Skill Encoding

In [ ]:
# ===== Step 3: Skill Encoding =====
unique_skills = data["skill"].unique()
skill2id = {skill: idx for idx, skill in enumerate(unique_skills)}

data["skill_id"] = data["skill"].map(skill2id)

NUM_SKILLS = len(skill2id)

print("Number of unique skills:", NUM_SKILLS)

Number of unique skills: 412


In [ ]:
# Build skill → content_source mapping

skill_to_module = (
    data[["skill", "content_source"]]
    .drop_duplicates()
    .set_index("skill")["content_source"]
    .to_dict()
)

# Create a mapping from skill ID to skill name
id2skill = {idx: skill for skill, idx in skill2id.items()}

# Create a mapping from skill ID to content source module
skill_id_to_module = {
    skill_id: skill_to_module[id2skill[skill_id]]
    for skill_id in id2skill.keys()
    if skill_id in id2skill and id2skill[skill_id] in skill_to_module
}

print("Total skills mapped (by name):", len(skill_to_module))
print("Total skills mapped (by ID):", len(skill_id_to_module))

Total skills mapped (by name): 412
Total skills mapped (by ID): 412


In [ ]:
pdets["mean_time_on_task"] = pdets["mean_time_on_task"].fillna(
    pdets["mean_time_on_task"].mean()
)
print("Final cleaned data shape:", data.shape)

Final cleaned data shape: (1045946, 21)


# Group By Student

In [ ]:
from collections import defaultdict

student_sequences = defaultdict(lambda: {"skills": [], "correct": []})

for row in data.itertuples():
    student_sequences[row.student_id]["skills"].append(row.skill_id)
    student_sequences[row.student_id]["correct"].append(row.correct)

print("Number of students:", len(student_sequences))

Number of students: 45907


# Create Training Samples

In [ ]:
MAX_SEQ = 100

In [ ]:
import torch

X_skills = []
X_correct = []
y_correctness = []  # Explicitly store correctness
y_next_skill = []   # New: store the actual next skill ID

for student in student_sequences:
    skills = student_sequences[student]["skills"]
    correct = student_sequences[student]["correct"]

    for i in range(1, len(skills)):
        start = max(0, i - MAX_SEQ)

        X_skills.append(skills[start:i])
        X_correct.append(correct[start:i])
        y_correctness.append(correct[i])  # next correctness
        y_next_skill.append(skills[i])    # New: actual next skill ID

Pad Sequences

In [ ]:
from torch.nn.utils.rnn import pad_sequence

X_skills = [torch.tensor(seq) for seq in X_skills]
X_correct = [torch.tensor(seq) for seq in X_correct]
y_correctness = torch.tensor(y_correctness) # Updated variable name
y_next_skill = torch.tensor(y_next_skill)   # New tensor for next skill

X_skills = pad_sequence(X_skills, batch_first=True)
X_correct = pad_sequence(X_correct, batch_first=True)

print("Skill tensor shape:", X_skills.shape)
print("Correct tensor shape:", X_correct.shape)
print("Target correctness shape:", y_correctness.shape) # Updated print
print("Target next skill shape:", y_next_skill.shape)    # New print

Skill tensor shape: torch.Size([1000039, 100])
Correct tensor shape: torch.Size([1000039, 100])
Target correctness shape: torch.Size([1000039])
Target next skill shape: torch.Size([1000039])


# SAKT Model

In [ ]:
class SAKT(nn.Module):
    def __init__(self, num_skills, embed_dim=128, num_heads=8, dropout=0.2):
        super(SAKT, self).__init__()

        self.skill_embed = nn.Embedding(num_skills + 1, embed_dim)
        self.response_embed = nn.Embedding(2, embed_dim)
        self.pos_embed = nn.Embedding(100, embed_dim)

        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)

        self.fc = nn.Linear(embed_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, skills, responses, return_state=False):
        seq_len = skills.size(1)
        positions = torch.arange(seq_len).unsqueeze(0).to(skills.device)

        skill_emb = self.skill_embed(skills)
        response_emb = self.response_embed(responses)

        x = skill_emb + response_emb + self.pos_embed(positions)
        x = x.permute(1, 0, 2)

        attn_output, _ = self.attention(x, x, x)
        attn_output = attn_output.permute(1, 0, 2)

        state = attn_output[:, -1, :]   # 128-dim student state

        out = self.fc(state)
        prob = torch.sigmoid(out).squeeze()

        if return_state:
            return prob, state
        return prob

Initialize Model

In [ ]:
torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Model ready")

Model ready


In [ ]:
# Save trained SAKT model
torch.save(model.state_dict(), "sakt_trained_model.pth")
print("SAKT model saved successfully.")

SAKT model saved successfully.


Loss & Optimizer

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Create Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_s_train, X_s_val, X_c_train, X_c_val, y_corr_train, y_corr_val, y_skill_train, y_skill_val = train_test_split(
    X_skills, X_correct, y_correctness, y_next_skill, # Split both correctness and next skill
    test_size=0.2,
    random_state=42
)

print("Train size:", X_s_train.shape)
print("Validation size:", X_s_val.shape)

Train size: torch.Size([800031, 100])
Validation size: torch.Size([200008, 100])


# Create DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 256

train_dataset = TensorDataset(X_s_train, X_c_train, y_corr_train, y_skill_train) # Add y_skill_train
val_dataset   = TensorDataset(X_s_val, X_c_val, y_corr_val, y_skill_val)     # Add y_skill_val

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Add AUC Calculation

In [ ]:
from sklearn.metrics import roc_auc_score
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for skills, correct, target_correctness, _ in loader: # Unpack 4 values, ignore the last one
            skills = skills.to(device)
            correct = correct.to(device)

            preds = model(skills, correct)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target_correctness.numpy())

    auc = roc_auc_score(all_targets, all_preds)
    return auc

# Train + Validate Each Epoch

In [ ]:
model = SAKT(num_skills=NUM_SKILLS).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for skills, correct, target, _ in train_loader:
        skills = skills.to(device)
        correct = correct.to(device)
        target = target.float().to(device)

        preds = model(skills, correct)
        loss = criterion(preds, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_auc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {total_loss/len(train_loader):.4f}")
    print(f"Validation AUC: {val_auc:.4f}")

Epoch 1
Train Loss: 0.5767
Validation AUC: 0.7166
Epoch 2
Train Loss: 0.5708
Validation AUC: 0.7193
Epoch 3
Train Loss: 0.5682
Validation AUC: 0.7235
Epoch 4
Train Loss: 0.5663
Validation AUC: 0.7235
Epoch 5
Train Loss: 0.5651
Validation AUC: 0.7233


# After training completes

In [ ]:
torch.save(model.state_dict(), "sakt_trained.pth")
print("Model saved successfully")

Model saved successfully


# Freeze SAKT

In [ ]:
for param in model.parameters():
    param.requires_grad = False

model.eval()

SAKT(
  (skill_embed): Embedding(413, 128)
  (response_embed): Embedding(2, 128)
  (pos_embed): Embedding(100, 128)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [ ]:
from google.colab import files
files.download("sakt_trained.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.upload()

Saving sakt_trained.pth to sakt_trained.pth


{'sakt_trained.pth': b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x15\x00\r\x00sakt_trained/data.pklFB\t\x00ZZZZZZZZZ\x80\x02ccollections\nOrderedDict\nq\x00)Rq\x01(X\x12\x00\x00\x00skill_embed.weightq\x02ctorch._utils\n_rebuild_tensor_v2\nq\x03((X\x07\x00\x00\x00storageq\x04ctorch\nFloatStorage\nq\x05X\x01\x00\x00\x000q\x06X\x03\x00\x00\x00cpuq\x07M\x80\xcetq\x08QK\x00M\x9d\x01K\x80\x86q\tK\x80K\x01\x86q\n\x89h\x00)Rq\x0btq\x0cRq\rX\x15\x00\x00\x00response_embed.weightq\x0eh\x03((h\x04h\x05X\x01\x00\x00\x001q\x0fh\x07M\x00\x01tq\x10QK\x00K\x02K\x80\x86q\x11K\x80K\x01\x86q\x12\x89h\x00)Rq\x13tq\x14Rq\x15X\x10\x00\x00\x00pos_embed.weightq\x16h\x03((h\x04h\x05X\x01\x00\x00\x002q\x17h\x07M\x002tq\x18QK\x00KdK\x80\x86q\x19K\x80K\x01\x86q\x1a\x89h\x00)Rq\x1btq\x1cRq\x1dX\x18\x00\x00\x00attention.in_proj_weightq\x1eh\x03((h\x04h\x05X\x01\x00\x00\x003q\x1fh\x07M\x00\xc0tq QK\x00M\x80\x01K\x80\x86q!K\x80K\x01\x86q"\x89h\x00)Rq#tq$Rq%X\x16

In [ ]:
model = SAKT(num_skills=412).to(device)
model.load_state_dict(torch.load("sakt_trained.pth", map_location=device))
model.eval()

for param in model.parameters():
    param.requires_grad = False

print("SAKT loaded successfully")

SAKT loaded successfully


In [ ]:
import os
print(os.listdir())

['.config', 'sakt_trained.pth', 'data', 'may_to_june.zip', 'sample_data']


# Extract Student State Representation

In [ ]:
print(len(skill2id))

412


Build Dueling Double DQN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DuelingDQN(nn.Module):
    def __init__(self, state_dim=128, action_dim=412):
        super(DuelingDQN, self).__init__()

        self.feature = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        # Value stream
        self.value = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        # Advantage stream
        self.advantage = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        x = self.feature(x)

        value = self.value(x)
        advantage = self.advantage(x)

        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q

Initialize Networks

In [ ]:
dqn = DuelingDQN().to(device)
target_dqn = DuelingDQN().to(device)

target_dqn.load_state_dict(dqn.state_dict())
target_dqn.eval()

DuelingDQN(
  (feature): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
  )
  (value): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (advantage): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=412, bias=True)
  )
)

Replay Buffer

In [ ]:
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            torch.stack(states),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float),
            torch.stack(next_states),
            torch.tensor(dones, dtype=torch.float)
        )

    def __len__(self):
        return len(self.buffer)

# Hyperparameters

In [ ]:
BATCH_SIZE = 64
GAMMA = 0.9
LR = 1e-4
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.995
TARGET_UPDATE = 1000

Optimizer

In [ ]:
optimizer = torch.optim.Adam(dqn.parameters(), lr=LR)
memory = ReplayBuffer()
epsilon = EPSILON_START
step_count = 0

In [ ]:
batch = next(iter(train_loader))
print(len(batch))

4


# Epsilon-Greedy Action Selection

In [ ]:
def select_action(state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, 411)  # 412 skills
    else:
        with torch.no_grad():
            q_values = dqn(state.unsqueeze(0))
            return q_values.argmax().item()

DQN Setup

In [ ]:
NUM_SKILLS = 412
STATE_DIM = 128   # from SAKT embedding

dqn = DuelingDQN(STATE_DIM, NUM_SKILLS).to(device)
target_dqn = DuelingDQN(STATE_DIM, NUM_SKILLS).to(device)

target_dqn.load_state_dict(dqn.state_dict())
target_dqn.eval()

DuelingDQN(
  (feature): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
  )
  (value): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (advantage): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=412, bias=True)
  )
)

In [ ]:
dummy = torch.randn(4, STATE_DIM).to(device)
print(dqn(dummy).shape)

torch.Size([4, 412])


# Build Replay Memory From Real Data

In [ ]:
def build_state(skills_hist, correct_hist):
    """
    Builds a state representation using the SAKT model.
    Args:
        skills_hist (list): List of skill IDs in the history.
        correct_hist (list): List of correctness (0/1) in the history.
    Returns:
        torch.Tensor: A 128-dimensional state embedding.
    """
    # Ensure history is not longer than MAX_SEQ
    hist_skills = torch.tensor(skills_hist, dtype=torch.long)[-MAX_SEQ:]
    hist_correct = torch.tensor(correct_hist, dtype=torch.long)[-MAX_SEQ:]

    # Pad to MAX_SEQ for SAKT model input
    padded_skills = torch.zeros(MAX_SEQ, dtype=torch.long)
    padded_correct = torch.zeros(MAX_SEQ, dtype=torch.long)
    padded_skills[-len(hist_skills):] = hist_skills
    padded_correct[-len(hist_correct):] = hist_correct

    with torch.no_grad():
        _, state_embedding = model(
            padded_skills.unsqueeze(0).to(device),
            padded_correct.unsqueeze(0).to(device),
            return_state=True
        )
    return state_embedding.squeeze(0).cpu()

# Ensure model is in eval mode
model.eval()

print("build_state function defined.")

build_state function defined.


In [ ]:
memory = ReplayBuffer(capacity=30000)

# Iterate through each student's interaction sequence
for student_id, seq_data in student_sequences.items():

    skills_full_seq = seq_data["skills"]
    correct_full_seq = seq_data["correct"]

    # Need at least two interactions
    if len(skills_full_seq) < 2:
        continue

    for t in range(len(skills_full_seq) - 1):

        # Current state (history up to time t)
        state = build_state(
            skills_full_seq[:t+1],
            correct_full_seq[:t+1]
        )

        # Action taken at time t
        action = skills_full_seq[t]

        # True next skill and correctness
        true_next_skill = skills_full_seq[t + 1]
        correct_next = correct_full_seq[t + 1]

        # -------- REWARD FUNCTION --------
        if action == true_next_skill:
            reward = 2 if correct_next == 1 else 1
        else:
            reward = -1
        # ---------------------------------

        # Next state (history including next interaction)
        next_state = build_state(
            skills_full_seq[:t+2],
            correct_full_seq[:t+2]
        )

        done = False

        memory.push(state, action, reward, next_state, done)

print("Replay buffer size:", len(memory.buffer))

Replay buffer size: 30000


In [ ]:
actions = [a for (_, a, _, _, _) in memory.buffer]
print("Unique actions:", len(set(actions)))

Unique actions: 330


# Double DQN Training Step

In [ ]:
def train_step():
    if len(memory) < BATCH_SIZE:
        return None

    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    states = states.to(device)
    next_states = next_states.to(device)
    actions = actions.long().to(device)
    rewards = rewards.float().to(device)
    dones = dones.float().to(device)

    # Current Q
    q_values = dqn(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Double DQN
    with torch.no_grad():
        next_actions = dqn(next_states).argmax(1)
        next_q = target_dqn(next_states).gather(
            1, next_actions.unsqueeze(1)
        ).squeeze(1)

        target = rewards + GAMMA * next_q * (1 - dones)

    loss = F.smooth_l1_loss(q_values, target)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dqn.parameters(), 5)
    optimizer.step()

    return loss.item()

In [ ]:
states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

print(type(states))
print(states.shape)

<class 'torch.Tensor'>
torch.Size([64, 128])


Training Loop

In [ ]:
losses = []

for step in range(5000):

    loss = train_step()

    if loss is not None:
        losses.append(loss)

    # Update target network
    if step % 100 == 0:
        target_dqn.load_state_dict(dqn.state_dict())

    # 🔎 Monitor Q-value range
    if step % 500 == 0:
        with torch.no_grad():
            sample_states, _, _, _, _ = memory.sample(BATCH_SIZE)
            sample_states = sample_states.to(device)
            q_vals = dqn(sample_states)

            print(f"Step {step} | Loss: {loss:.4f}")
            print("Q range:", q_vals.min().item(), q_vals.max().item())

Step 0 | Loss: 0.9932
Q range: -0.37900757789611816 0.26401054859161377
Step 500 | Loss: 1.1272
Q range: -0.38233914971351624 0.25653740763664246
Step 1000 | Loss: 0.9195
Q range: -0.3612740933895111 0.28562119603157043
Step 1500 | Loss: 1.0299
Q range: -0.35164085030555725 0.26859742403030396
Step 2000 | Loss: 1.1099
Q range: -0.3742518424987793 0.27558228373527527
Step 2500 | Loss: 1.0521
Q range: -0.3617379367351532 0.27041417360305786
Step 3000 | Loss: 0.9935
Q range: -0.38707417249679565 0.281832754611969
Step 3500 | Loss: 0.9625
Q range: -0.3564785122871399 0.26700982451438904
Step 4000 | Loss: 1.0580
Q range: -0.3831198811531067 0.3089087903499603
Step 4500 | Loss: 1.0846
Q range: -0.38233914971351624 0.2729582190513611


# Evaluation Function

In [ ]:
import torch
import math

def evaluate_model(student_sequences, K=5):
    dqn.eval()
    model.eval()

    total = 0
    hit_count = 0
    ndcg_sum = 0

    with torch.no_grad():
        for student_id, seq_data in student_sequences.items():
            skills = seq_data["skills"]
            correct = seq_data["correct"]

            if len(skills) < 2:
                continue

            skills_tensor = torch.tensor(skills, dtype=torch.long)
            correct_tensor = torch.tensor(correct, dtype=torch.long)

            for t in range(1, len(skills)):

                # Build state history (s_t)
                hist_skills = skills_tensor[:t][-MAX_SEQ:]
                hist_correct = correct_tensor[:t][-MAX_SEQ:]

                padded_skills = torch.zeros(MAX_SEQ, dtype=torch.long)
                padded_correct = torch.zeros(MAX_SEQ, dtype=torch.long)

                padded_skills[-len(hist_skills):] = hist_skills
                padded_correct[-len(hist_correct):] = hist_correct

                # Get state embedding
                _, state = model(
                    padded_skills.unsqueeze(0).to(device),
                    padded_correct.unsqueeze(0).to(device),
                    return_state=True
                )

                state = state.to(device)

                # Get Q-values
                q_values = dqn(state).squeeze(0)

                # Top-K predictions
                topk = torch.topk(q_values, K).indices.cpu().tolist()

                true_skill = skills[t]

                total += 1

                # Hit@K
                if true_skill in topk:
                    hit_count += 1

                    # NDCG@K
                    rank = topk.index(true_skill)
                    ndcg_sum += 1 / math.log2(rank + 2)

    hit_at_k = hit_count / total
    ndcg_at_k = ndcg_sum / total

    return hit_at_k, ndcg_at_k

#Split Student Sequences for Training and Validation

In [ ]:
from sklearn.model_selection import train_test_split

student_ids = list(student_sequences.keys())

train_student_ids, val_student_ids = train_test_split(
    student_ids,
    test_size=0.2,
    random_state=42
)

student_sequences_train = {sid: student_sequences[sid] for sid in train_student_ids}
student_sequences_val = {sid: student_sequences[sid] for sid in val_student_ids}

print("Number of students in training set:", len(student_sequences_train))
print("Number of students in validation set:", len(student_sequences_val))

Number of students in training set: 36725
Number of students in validation set: 9182


In [ ]:
hit5, ndcg5 = evaluate_model(student_sequences_val, K=5)

print("Hit@5:", hit5)
print("NDCG@5:", ndcg5)

Hit@5: 0.014584746827490891
NDCG@5: 0.0077037622636942195


In [ ]:
data.groupby("content_source")["skills"].nunique()

,skills
content_source,
"['Certified Content', 'Engage New York']",1
"['Certified Content', 'Skill Builder']",9
"['Certified Content', 'State Tests']",33
['Certified Content'],268
"['Engage New York', 'Skill Builder']",2
['Engage New York'],477
['Eureka Math'],39
"['Illustrative Mathematics', 'Open Up Resources']",279
['Illustrative Mathematics'],6


In [ ]:
def module_hit_at_k(topk_preds, true_skill_ids, skill_id_to_module):
    batch_size, K = topk_preds.shape
    hits = 0

    for i in range(batch_size):
        true_skill_id = true_skill_ids[i].item()
        true_module = skill_id_to_module.get(true_skill_id, None)

        if true_module is None:
            continue

        predicted_skill_ids = topk_preds[i].tolist()

        for predicted_skill_id in predicted_skill_ids:
            if skill_id_to_module.get(predicted_skill_id, None) == true_module:
                hits += 1
                break  # count only once per sample

    return hits / batch_size

In [ ]:
module_recall_at_k = module_hit_at_k

In [ ]:
import numpy as np

def module_ndcg_at_k(topk_preds, true_skill_ids, skill_id_to_module):
    batch_size, K = topk_preds.shape
    ndcg_total = 0

    for i in range(batch_size):
        true_skill_id = true_skill_ids[i].item()
        true_module = skill_id_to_module.get(true_skill_id, None)

        if true_module is None:
            continue

        dcg = 0

        for rank, predicted_skill_id in enumerate(topk_preds[i].tolist()):
            if skill_id_to_module.get(predicted_skill_id, None) == true_module:
                dcg = 1 / np.log2(rank + 2)
                break

        idcg = 1.0  # ideal case at rank 1 (if true module is found at rank 1)
        ndcg_total += dcg / idcg

    return ndcg_total / batch_size

In [ ]:
model.eval()

def hit_at_k(topk_preds, true_skill_ids, k=5):

    hits = 0
    for i in range(len(true_skill_ids)):
        if true_skill_ids[i].item() in topk_preds[i, :k].cpu().tolist():
            hits += 1
    return hits / len(true_skill_ids)

all_skill_hits = []
all_module_hits = []

with torch.no_grad():

    for hist_skills_batch, hist_correct_batch, true_correctness_batch, true_skill_batch in val_loader: # Unpack 4 values

        hist_skills_batch = hist_skills_batch.to(device)
        hist_correct_batch = hist_correct_batch.to(device)
        # true_correctness_batch is not used for skill prediction evaluation, but it's part of the loader
        true_skill_batch = true_skill_batch.to(device)

        # Get SAKT state embeddings
        _, sakt_states = model(
            hist_skills_batch,
            hist_correct_batch,
            return_state=True
        )

        # 1️⃣ Get model predictions (DQN Q-values for these states)
        logits = dqn(sakt_states)

        # 2️⃣ Get Top-K predicted skills
        topk = torch.topk(logits, k=5, dim=1).indices

        # 3️⃣ Compute skill-level metrics
        skill_hit = hit_at_k(topk, true_skill_batch, k=5)

        # 4️⃣ Compute module-level metrics ⭐ (THIS IS STEP 5)
        # Assuming module_hit_at_k is updated to take skill_id_to_module
        module_hit = module_hit_at_k(topk, true_skill_batch, skill_id_to_module)

        # 5️⃣ Store results
        all_skill_hits.append(skill_hit)
        all_module_hits.append(module_hit)

# 6️⃣ Final average
print("Skill Hit@5:", sum(all_skill_hits) / len(all_skill_hits))
print("Module Hit@5:", sum(all_module_hits) / len(all_module_hits))

Skill Hit@5: 0.013676870204603581
Module Hit@5: 0.3984375


In [ ]:
model.eval()

all_hit5 = []
all_hit10 = []
# all_recall10 = [] # Removed as recall_at_k is not defined
all_module_hit = []

with torch.no_grad():

    # Unpack 4 values from val_loader, consistent with pALe5tYamCEZ
    for hist_skills_batch, hist_correct_batch, true_correctness_batch, true_skill_batch in val_loader:

        hist_skills_batch = hist_skills_batch.to(device)
        hist_correct_batch = hist_correct_batch.to(device)
        true_skill_batch = true_skill_batch.to(device) # This is the target skill

        # Get SAKT state embeddings
        _, sakt_states = model(
            hist_skills_batch,
            hist_correct_batch,
            return_state=True
        )

        # Pass SAKT states to DQN for Q-value predictions
        logits = dqn(sakt_states)

        # Top-10 predictions
        top10 = torch.topk(logits, k=10, dim=1).indices

        # Skill metrics
        hit5 = hit_at_k(top10, true_skill_batch, 5)
        hit10 = hit_at_k(top10, true_skill_batch, 10)
        # recall10 = recall_at_k(top10, true_skill_batch, 10) # Removed

        # Module metric
        module_hit = module_hit_at_k(top10, true_skill_batch, skill_id_to_module)

        all_hit5.append(hit5)
        all_hit10.append(hit10)
        # all_recall10.append(recall10) # Removed
        all_module_hit.append(module_hit)

print("Skill Hit@5:", sum(all_hit5)/len(all_hit5))
print("Skill Hit@10:", sum(all_hit10)/len(all_hit10))
# print("Recall@10:", sum(all_recall10)/len(all_recall10)) # Removed
print("Module Hit@10:", sum(all_module_hit)/len(all_module_hit))

Skill Hit@5: 0.013676870204603581
Skill Hit@10: 0.035084097044614945
Module Hit@10: 0.6460298113810742


# Function to Recommend Modules

In [ ]:
def recommend_modules_for_student(student_skills_history, student_correctness_history, k_top_skills=10):
    model.eval()  # SAKT in evaluation mode
    dqn.eval()    # DQN in evaluation mode

    # Prepare student history for SAKT input
    hist_skills = torch.tensor(student_skills_history, dtype=torch.long)
    hist_correct = torch.tensor(student_correctness_history, dtype=torch.long)

    # Ensure history is not longer than MAX_SEQ
    hist_skills = hist_skills[-MAX_SEQ:]
    hist_correct = hist_correct[-MAX_SEQ:]

    # Pad to MAX_SEQ for SAKT model input
    padded_skills = torch.zeros(MAX_SEQ, dtype=torch.long)
    padded_correct = torch.zeros(MAX_SEQ, dtype=torch.long)
    padded_skills[-len(hist_skills):] = hist_skills
    padded_correct[-len(hist_correct):] = hist_correct

    with torch.no_grad():
        # Get student state embedding from SAKT
        _, student_state = model(
            padded_skills.unsqueeze(0).to(device),
            padded_correct.unsqueeze(0).to(device),
            return_state=True
        )

        # Get Q-values for all skills from DQN
        q_values = dqn(student_state).squeeze(0)

        # Get top K skill indices
        top_k_skill_ids = torch.topk(q_values, k=k_top_skills).indices.cpu().tolist()

    # Map top K skill IDs to their modules
    recommended_modules = set()
    for skill_id in top_k_skill_ids:
        module = skill_id_to_module.get(skill_id)
        if module: # Ensure module is not None
            recommended_modules.add(module)

    return list(recommended_modules)


### Recommendations for a selection of existing students

In [ ]:
import random

# Let's pick a few more random student IDs from the validation set
# You can change the number of students to sample or iterate through specific IDs
sample_student_ids = random.sample(val_student_ids, 5) # Get 5 random student IDs from validation set

for student_id in sample_student_ids:
    student_history = student_sequences[student_id]

    # Use the full history for a more realistic scenario
    current_skills_history = student_history["skills"]
    current_correctness_history = student_history["correct"]

    print(f"\n--- Student ID: {student_id} ---")
    print(f"Student's total history length: {len(current_skills_history)}")

    if len(current_skills_history) == 0:
        print("No history for this student, cannot provide recommendations.")
        continue

    recommended = recommend_modules_for_student(
        current_skills_history,
        current_correctness_history,
        k_top_skills=10 # Using K=10 as before
    )

    print("Recommended Modules:")
    if recommended:
        for module in recommended:
            print(f"- {module}")
    else:
        print("No specific module recommendations (could be due to lack of skills mapped to modules).")



--- Student ID: 929713 ---
Student's total history length: 35
Recommended Modules:
- ['Undetermined']
- ['Skill Builder']
- ['Utah Math']
- ['Open Up Resources']
- ['State Tests']

--- Student ID: 42732 ---
Student's total history length: 8
Recommended Modules:
- ['Undetermined']
- ['Illustrative Mathematics', 'Open Up Resources']
- ['Skill Builder']
- ['Utah Math']
- ['State Tests']

--- Student ID: 564500 ---
Student's total history length: 1
Recommended Modules:
- ['Illustrative Mathematics', 'Open Up Resources']
- ['Skill Builder']
- ['Utah Math']
- ['Engage New York']
- ['State Tests']
- ['Certified Content']

--- Student ID: 235159 ---
Student's total history length: 4
Recommended Modules:
- ['Illustrative Mathematics', 'Open Up Resources']
- ['Skill Builder']
- ['Utah Math']
- ['Engage New York']
- ['State Tests']
- ['Certified Content']

--- Student ID: 949825 ---
Student's total history length: 87
Recommended Modules:
- ['Illustrative Mathematics', 'Open Up Resources']
- ['Sk